In [1]:
import pandas as pd
import numpy as np

#orginele dataset inlezen
df = pd.read_csv('Leesmateriaal.csv')


In [2]:
#Alle overige kolommen verwijderen die niet relevant zijn voor de analyse
df = df.drop(columns=
             ['Unnamed: 3', 'materiaal_id', 'Themas_boek', 'woordenaantal_omschrijving', 
              'gem_woordlengte', 'gem_zinslengte_woorden', 'aandeel_lange_woorden_pct', 
              'aantal_zinnen', 'interne_batchcode', 'administratieve_prioriteit', 
              'genre_kopie', 'Gemiddelde beoordeling'])

In [3]:
# aantal unieke en missende waarden en datatypes
print("Aantal rijen en kolommen:", df.shape)
overzicht = pd.DataFrame({
    "Aantal unieke waarden": df.nunique(),
    "Aantal missende waarden": df.isna().sum(),
    "Datatype": df.dtypes,
    "Aantal dubbele waarden": df.apply(
        lambda kolom: kolom.dropna().duplicated().sum()
    )
})
 
display(overzicht)

Aantal rijen en kolommen: (283, 21)


,Aantal unieke waarden,Aantal missende waarden,Datatype,Aantal dubbele waarden
Titel,277,0,str,6
Auteur,231,7,str,45
Korte omschrijving,282,0,str,1
Type materiaal,10,1,str,272
Niveau,4,6,str,273
genre,3,26,str,254
onderwerp,47,181,str,55
trefwoorden,93,181,str,9
taal,4,0,str,279
aantal_paginas,159,17,float64,107


In [4]:
df["aantal_paginas"] = df["aantal_paginas"].mask(
    df["aantal_paginas"] <= 0,
    np.nan
)

In [5]:
df["taal"] = (
    df["taal"]
    .str.strip()
    .str.lower()
    .replace({
        "ned": "Nederlands",
        "nl": "Nederlands",
        "nederlands": "Nederlands",
        "dutch": "Nederlands"
    })
)

In [6]:
df["Auteur"] = (
    df["Auteur"]
    .str.replace(r"\s*\(redactie\)", "", regex=True)
    .str.strip()
)

df.iloc[139:143]

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,genre,onderwerp,trefwoorden,taal,aantal_paginas,...,bron_url,doelgroep,gegevensstatus,leescomplexiteit_score,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode,doelgroep_extra
139,Vanaf 2 augustus wordt duidelijker of het AI i...,Autoriteit Persoonsgegevens,"AI, privacy, deepfakes",online artikel,3F,informatief,NaN,NaN,Nederlands,2.0,...,https://www.autoriteitpersoonsgegevens.nl/actu...,MBO,Aangevuld (deels gesimuleerd),33.3,verhalen,Leeswereld,NaN,NaN,CAT-00140,MBO
140,Transparantie-eisen voor AI: wat betekent dit ...,Autoriteit Persoonsgegevens,"AI-verordening, rechten van burgers",online artikel,3F+,informatief,NaN,NaN,Nederlands,6.0,...,https://www.autoriteitpersoonsgegevens.nl/them...,MBO,Aangevuld (deels gesimuleerd),30.7,verhalen,NaN,NaN,online,CAT-00141,MBO
141,AI-verordening,Autoriteit Persoonsgegevens,Europese AI-wetgeving,online artikel,3F+,informatief,NaN,NaN,Nederlands,6.0,...,https://www.autoriteitpersoonsgegevens.nl/them...,MBO,gecontroleerd,49.0,verhalen,Educatief Media,NaN,online,CAT-00142,MBO
142,"AP: AI-Impactbarometer kleurt rood, actie is n...",Autoriteit Persoonsgegevens,"AI, toezicht, discriminatie, recht",online artikel,3F+,NaN,NaN,NaN,Nederlands,4.0,...,https://www.autoriteitpersoonsgegevens.nl/actu...,MBO,Aangevuld (deels gesimuleerd),44.7,verhalen,Educatief Media,2021,NaN,CAT-00143,MBO/NT2


In [7]:
df["interessecluster"] = (
    df["interessecluster"]
    .str.strip()
    .str.lower()
    .str.capitalize()
)
df["interessecluster"].value_counts()

interessecluster
Verhalen        268
Maatschappij     13
Vrije tijd        2
Name: count, dtype: int64

In [8]:
def merge_doelgroep(row):
    doelgroepen = set()

    for kolom in ["doelgroep", "doelgroep_extra"]:
        waarde = row.get(kolom)

        if pd.notna(waarde):
            for doelgroep in str(waarde).split("/"):
                doelgroep = doelgroep.strip().upper()

                if doelgroep:
                    doelgroepen.add(doelgroep)

    return " / ".join(sorted(doelgroepen))


df["doelgroep"] = df.apply(merge_doelgroep, axis=1)

df = df.drop(columns=["doelgroep_extra"])

In [9]:
df["doelgroep"] = df["doelgroep"].replace("", np.nan)

In [10]:
df["doelgroep"].value_counts()

doelgroep
MBO / NT2    212
MBO           65
NT2            3
Name: count, dtype: int64

In [11]:
df["Type materiaal"].value_counts()

Type materiaal
Boek               140
Boek - thriller     37
online artikel      34
Boek - roman        28
dichtbundel         23
blogpost            15
tijdschrift          2
boek                 1
krant (papier)       1
digitale krant       1
Name: count, dtype: int64

In [12]:
def verwerk_type_materiaal(df):
    masker = (
        df["Type materiaal"]
        .fillna("")
        .str.strip()
        .str.lower()
        .str.startswith("boek -")
    )

    for index in df.index[masker]:
        extra_genre = df["Type materiaal"].loc[index].split("-", 1)[1].strip().capitalize()
        bestaand_genre = df.at[index, "genre"]

        if pd.isna(bestaand_genre) or str(bestaand_genre).strip() == "":
            df.at[index, "genre"] = extra_genre
        elif extra_genre.lower() not in str(bestaand_genre).lower():
            df.at[index, "genre"] = f"{bestaand_genre} / {extra_genre}"

    # Maak bijvoorbeeld "Boek - thriller" gelijk aan "Boek"
    df.loc[masker, "Type materiaal"] = "Boek"

    # Type materiaal normaliseren
    df["Type materiaal"] = (
        df["Type materiaal"]
        .astype("string")
        .str.strip()
        .str.lower()
        .replace({
            "digitale krant": "krant",
            "krant (papier)": "krant"
        })
    )

    # Boek met of zonder genre wordt Boek
    df.loc[
        df["Type materiaal"].str.match(r"^boek(?:\s*-.*)?$", na=False),
        "Type materiaal"
    ] = "boek"

    # Alleen de eerste letter hoofdletter maken
    df["Type materiaal"] = df["Type materiaal"].str.capitalize()
        
    return df

df = verwerk_type_materiaal(df)
df["Type materiaal"].value_counts()

Type materiaal
Boek              206
Online artikel     34
Dichtbundel        23
Blogpost           15
Tijdschrift         2
Krant               2
Name: count, dtype: Int64

In [13]:
df["beschikbaar_formaat"].value_counts()

beschikbaar_formaat
Papier      66
PDF         54
online      39
Digitaal    38
papier      30
Online      13
Name: count, dtype: int64

In [14]:
df["beschikbaar_formaat"] = (
    df["beschikbaar_formaat"]
    .str.strip()
    .str.lower()
)

df["beschikbaar_formaat"] = df["beschikbaar_formaat"].replace("digitaal", "online")
df["beschikbaar_formaat"] = df["beschikbaar_formaat"].str.capitalize()

df["beschikbaar_formaat"].value_counts()

beschikbaar_formaat
Papier    96
Online    90
Pdf       54
Name: count, dtype: int64

In [15]:
df["gegevensstatus"] = df["gegevensstatus"].replace({
    "Aangevuld (deels gesimuleerd)": "Beschikbaar",
    "gecontroleerd": "Niet beschikbaar"
})

df["gegevensstatus"] = df["gegevensstatus"].fillna("Onbekend")

df["gegevensstatus"].value_counts()

gegevensstatus
Beschikbaar         245
Niet beschikbaar     19
In behandeling       11
Onbekend              8
Name: count, dtype: int64

In [16]:
df["publicatiejaar"].value_counts()

publicatiejaar
2023        44
2013        42
2025        31
2018        30
1998        26
2005        21
2021        16
onbekend    10
2099         7
Name: count, dtype: int64

In [17]:
from datetime import datetime

df["publicatiejaar"] = pd.to_numeric(
    df["publicatiejaar"],
    errors="coerce"
)

df.loc[
    df["publicatiejaar"] > datetime.now().year,
    "publicatiejaar"
] = np.nan

df["publicatiejaar"] = df["publicatiejaar"].replace({"onbekend": np.nan})

df["publicatiejaar"] = df["publicatiejaar"].astype("Int64")

df["publicatiejaar"].value_counts()

publicatiejaar
2023    44
2013    42
2025    31
2018    30
1998    26
2005    21
2021    16
Name: count, dtype: Int64

In [18]:
df["bron_url"] = (
    df["bron_url"]
    .str.replace(r"\s*\[.*$", "", regex=True)
    .str.strip()
)

In [19]:
df["onderwerp"] = df["onderwerp"].replace(
    r"^\[.*\]$",
    np.nan,
    regex=True
)

In [20]:
df["trefwoorden"] = df["trefwoorden"].replace(
    r"^\[.*\]$",
    np.nan,
    regex=True
)

In [21]:
# aantal unieke en missende waarden en datatypes
print("Aantal rijen en kolommen:", df.shape)
overzicht = pd.DataFrame({
    "Aantal unieke waarden": df.nunique(),
    "Aantal missende waarden": df.isna().sum(),
    "Datatype": df.dtypes,
    "Aantal dubbele waarden": df.apply(
        lambda kolom: kolom.dropna().duplicated().sum()
    )
})
 
display(overzicht)

Aantal rijen en kolommen: (283, 20)


,Aantal unieke waarden,Aantal missende waarden,Datatype,Aantal dubbele waarden
Titel,277,0,str,6
Auteur,230,7,str,46
Korte omschrijving,282,0,str,1
Type materiaal,6,1,string,276
Niveau,4,6,str,273
genre,7,20,str,256
onderwerp,46,183,str,54
trefwoorden,92,183,str,8
taal,1,0,str,282
aantal_paginas,157,28,float64,98


In [22]:
df.to_csv("Leesmateriaal_OW.csv", index=False)